## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten laden ──────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)  / 255.0

# ── 2. Modell-Fabrik ──────────────────────────────────────────────────────────
def modell_erstellen(dropout_rate=0.0, name="Kein_Dropout"):
    """
    Erstellt ein identisches Netzwerk – nur die Dropout-Rate variiert.
    dropout_rate=0.0 → kein Dropout
    """
    schichten = [
        tf.keras.layers.Dense(256, activation="relu", input_shape=(784,)),
        tf.keras.layers.Dense(128, activation="relu"),
    ]
    if dropout_rate > 0:
        schichten.insert(1, tf.keras.layers.Dropout(dropout_rate))
        schichten.append(tf.keras.layers.Dropout(dropout_rate))

    schichten.append(tf.keras.layers.Dense(10, activation="softmax"))

    m = tf.keras.Sequential(schichten, name=name)
    m.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

# ── 3. Drei Modelle trainieren ────────────────────────────────────────────────
EPOCHEN = 20
konfigurationen = [
    (0.0, "Kein Dropout"),
    (0.3, "Dropout 0.3"),
    (0.5, "Dropout 0.5"),
]

histories = []
for rate, name in konfigurationen:
    print(f"\nTrainiere: {name}...")
    m = modell_erstellen(dropout_rate=rate, name=name.replace(" ", "_"))
    hist = m.fit(
        x_train, y_train,
        epochs=EPOCHEN,
        batch_size=128,
        validation_split=0.15,
        verbose=0
    )
    test_loss, test_acc = m.evaluate(x_test, y_test, verbose=0)
    print(f"  Test-Genauigkeit: {test_acc:.4f}")
    histories.append((name, hist, test_acc))

# ── 4. Overfitting-Analyse ────────────────────────────────────────────────────
print("\n── Overfitting-Analyse ──")
print(f"{'Modell':<18} {'Train-Acc':>10} {'Val-Acc':>10} {'Diff (Überanpassung)':>22}")
print("-" * 63)
for name, hist, test_acc in histories:
    train_acc_final = hist.history["accuracy"][-1]
    val_acc_final   = hist.history["val_accuracy"][-1]
    diff            = train_acc_final - val_acc_final
    print(f"{name:<18} {train_acc_final:>10.4f} {val_acc_final:>10.4f} {diff:>22.4f}")

# ── 5. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
farben = ["#e74c3c", "#2ecc71", "#3498db"]

for (name, hist, test_acc), farbe in zip(histories, farben):
    # Training: durchgezogen, Validierung: gestrichelt
    axes[0].plot(hist.history["accuracy"],
                 color=farbe, linestyle="-",  label=f"{name} (Train)")
    axes[0].plot(hist.history["val_accuracy"],
                 color=farbe, linestyle="--", label=f"{name} (Val)")

axes[0].set_title("Genauigkeit: Training vs. Validierung\n"
                   "(durchg. = Training, gestrich. = Val)")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Genauigkeit")
axes[0].legend(fontsize=7)
axes[0].grid(True)

# Test-Genauigkeit Balkendiagramm
test_accs = [e[2] for e in histories]
axes[1].bar([e[0] for e in histories], test_accs, color=farben, edgecolor="black")
axes[1].set_title("Test-Genauigkeit Vergleich")
axes[1].set_ylabel("Genauigkeit")
axes[1].set_ylim(0.94, 1.0)
for i, acc in enumerate(test_accs):
    axes[1].text(i, acc + 0.0005, f"{acc:.4f}", ha="center", fontsize=10)
axes[1].grid(True, axis="y", alpha=0.3)

plt.suptitle("Dropout: Vergleich Kein Dropout vs. 0.3 vs. 0.5", fontsize=13)
plt.tight_layout()
plt.savefig("A8_1_dropout_vergleich.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A8_1_dropout_vergleich.png")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten ────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:8000].astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)         / 255.0
y_train = y_train[:8000]

# ── 2. Modell-Fabrik mit Regularisierung ─────────────────────────────────────
def modell_erstellen(regularisierung=None, name="Kein_Reg"):
    """
    regularisierung: None, 'l1', 'l2'
    """
    if regularisierung == "l1":
        reg = tf.keras.regularizers.l1(0.01)
    elif regularisierung == "l2":
        reg = tf.keras.regularizers.l2(0.01)
    else:
        reg = None

    m = tf.keras.Sequential([
        tf.keras.layers.Dense(
            256, activation="relu",
            kernel_regularizer=reg, input_shape=(784,), name="dense_1"
        ),
        tf.keras.layers.Dense(
            128, activation="relu",
            kernel_regularizer=reg, name="dense_2"
        ),
        tf.keras.layers.Dense(10, activation="softmax", name="ausgabe"),
    ], name=name)

    m.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

# ── 3. Alle drei Varianten trainieren ─────────────────────────────────────────
EPOCHEN = 25
varianten = [
    (None,  "Keine Regularisierung"),
    ("l1",  "L1-Regularisierung"),
    ("l2",  "L2-Regularisierung"),
]

histories = {}
modelle   = {}

for reg, name in varianten:
    print(f"\nTrainiere: {name}...")
    m = modell_erstellen(regularisierung=reg, name=name)
    hist = m.fit(
        x_train, y_train,
        epochs=EPOCHEN,
        batch_size=64,
        validation_split=0.15,
        verbose=0
    )
    test_loss, test_acc = m.evaluate(x_test, y_test, verbose=0)
    print(f"  Test-Genauigkeit: {test_acc:.4f}")
    histories[name] = hist
    modelle[name]   = m

# ── 4. Gewichtsverteilung analysieren ─────────────────────────────────────────
print("\n── Gewichtsstatistiken (dense_1) ──")
for reg, name in varianten:
    gewichte = modelle[name].get_layer("dense_1").get_weights()[0].flatten()
    print(f"{name:<28}: "
          f"Mittelwert={gewichte.mean():.4f}, "
          f"Std={gewichte.std():.4f}, "
          f"Max-Betrag={np.abs(gewichte).max():.4f}, "
          f"Nullnahe (<0.01)={np.sum(np.abs(gewichte)<0.01)/len(gewichte)*100:.1f}%")

# ── 5. Visualisierung ─────────────────────────────────────────────────────────
farben = ["#e74c3c", "#2ecc71", "#3498db"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (a) Verlustverläufe
for (reg, name), farbe in zip(varianten, farben):
    axes[0].plot(histories[name].history["loss"],
                 color=farbe, linestyle="-",  label=f"{name} (Train)")
    axes[0].plot(histories[name].history["val_loss"],
                 color=farbe, linestyle="--", label=f"{name} (Val)")
axes[0].set_title("Verlust (Loss)")
axes[0].set_xlabel("Epoche")
axes[0].legend(fontsize=7)
axes[0].grid(True)

# (b) Genauigkeitsverläufe
for (reg, name), farbe in zip(varianten, farben):
    axes[1].plot(histories[name].history["val_accuracy"],
                 color=farbe, label=name, linewidth=2)
axes[1].set_title("Validierungs-Genauigkeit")
axes[1].set_xlabel("Epoche")
axes[1].legend(fontsize=8)
axes[1].grid(True)

# (c) Gewichtsverteilung (Histogramm)
for (reg, name), farbe in zip(varianten, farben):
    gewichte = modelle[name].get_layer("dense_1").get_weights()[0].flatten()
    axes[2].hist(gewichte, bins=50, alpha=0.6, color=farbe, label=name, density=True)
axes[2].set_title("Gewichtsverteilung (dense_1)")
axes[2].set_xlabel("Gewichtswert")
axes[2].set_ylabel("Dichte")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle("L1 vs. L2 Regularisierung: Verlust, Genauigkeit, Gewichtsverteilung",
             fontsize=12)
plt.tight_layout()
plt.savefig("A8_2_l1_l2_regularisierung.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A8_2_l1_l2_regularisierung.png")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten ────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)  / 255.0

# ── 2. Modelle aufbauen ───────────────────────────────────────────────────────
def netzwerk_ohne_bn():
    """5-schichtiges Netzwerk OHNE Batch Normalization."""
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation="relu", input_shape=(784,)),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(10,  activation="softmax"),
    ], name="Ohne_BatchNorm")

def netzwerk_mit_bn():
    """5-schichtiges Netzwerk MIT Batch Normalization nach jeder Dense-Schicht."""
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, input_shape=(784,)),
        tf.keras.layers.BatchNormalization(),   # BN normalisiert Aktivierungen
        tf.keras.layers.Activation("relu"),

        tf.keras.layers.Dense(256),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),

        tf.keras.layers.Dense(256),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),

        tf.keras.layers.Dense(256),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),

        tf.keras.layers.Dense(256),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),

        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="Mit_BatchNorm")

# ── 3. Modelle kompilieren ────────────────────────────────────────────────────
def modell_kompilieren(modell):
    modell.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modell

m_ohne = modell_kompilieren(netzwerk_ohne_bn())
m_mit  = modell_kompilieren(netzwerk_mit_bn())

print("Ohne BN – Parameter:", m_ohne.count_params())
print("Mit BN  – Parameter:", m_mit.count_params())

# ── 4. Training ───────────────────────────────────────────────────────────────
EPOCHEN = 20

print("\nTrainiere Modell OHNE Batch Normalization...")
history_ohne = m_ohne.fit(
    x_train, y_train, epochs=EPOCHEN, batch_size=128,
    validation_split=0.1, verbose=0
)

print("Trainiere Modell MIT Batch Normalization...")
history_mit = m_mit.fit(
    x_train, y_train, epochs=EPOCHEN, batch_size=128,
    validation_split=0.1, verbose=0
)

# ── 5. Vergleich ──────────────────────────────────────────────────────────────
acc_ohne = m_ohne.evaluate(x_test, y_test, verbose=0)[1]
acc_mit  = m_mit.evaluate(x_test, y_test, verbose=0)[1]

print(f"\nTest-Genauigkeit OHNE BN: {acc_ohne:.4f}")
print(f"Test-Genauigkeit MIT  BN: {acc_mit:.4f}")

# Konvergenzgeschwindigkeit: Epoche bis 90% Val-Accuracy
def epoche_bis_schwellwert(hist, schwellwert=0.90):
    val_accs = hist.history["val_accuracy"]
    for i, acc in enumerate(val_accs):
        if acc >= schwellwert:
            return i + 1
    return None

ep_ohne = epoche_bis_schwellwert(history_ohne)
ep_mit  = epoche_bis_schwellwert(history_mit)
print(f"\nEpoche bis 90% Val-Acc – OHNE BN: {ep_ohne}")
print(f"Epoche bis 90% Val-Acc – MIT  BN: {ep_mit}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Verlust
axes[0].plot(history_ohne.history["val_loss"],
             label="OHNE BatchNorm", color="#e74c3c", linewidth=2)
axes[0].plot(history_mit.history["val_loss"],
             label="MIT BatchNorm",  color="#2ecc71", linewidth=2)
axes[0].set_title("Validierungs-Verlust")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Verlust")
axes[0].legend()
axes[0].grid(True)

# Genauigkeit
axes[1].plot(history_ohne.history["val_accuracy"],
             label="OHNE BatchNorm", color="#e74c3c", linewidth=2)
axes[1].plot(history_mit.history["val_accuracy"],
             label="MIT BatchNorm",  color="#2ecc71", linewidth=2)
axes[1].axhline(0.90, color="gray", linestyle="--", alpha=0.7, label="90%-Schwellwert")
axes[1].set_title("Validierungs-Genauigkeit")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Genauigkeit")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Batch Normalization: Schnellere Konvergenz und stabileres Training",
             fontsize=13)
plt.tight_layout()
plt.savefig("A8_3_batch_normalization.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A8_3_batch_normalization.png")
